Question 3: By using AWS Sagemaker analyze 20 scraped images and save results to a csv file
(10 po

In [1]:
# Analyze 20 images with Amazon Rekognition (SageMaker Notebook Instance; no functions)
# Objective:
# Load Q1 data (Excel), select 20 image URLs, call Rekognition DetectLabels,
# produce a structured results file (Excel) that includes top label, confidence, and filtered label list.

# Imports
import os
import json
import time
import random
import sys
import subprocess

import pandas as pd
import requests
import boto3


# Configuration
FILE_PATH = "214_Q1.xlsx"           # Excel file generated from Q1 output
SAMPLE_SIZE = 20                    # number of images to analyze
SAMPLE_METHOD = "first"             # "first" or "random"
CONF_THRESHOLD = 90.0               # minimum confidence (%) to retain labels in labels_json
REQUEST_TIMEOUT = 20                # HTTP request timeout (seconds)
SLEEP_BETWEEN = (0.5, 1.0)          # Controlled delay interval between HTTP requests (in seconds)
OUTPUT_XLSX = "q3_image_analysis.xlsx"

# Load dataset and normalize columns
# If an Excel file is provided, ensure the engine is available
if FILE_PATH.lower().endswith((".xlsx", ".xls")):
    try:
        import openpyxl  # noqa
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    df = pd.read_excel(FILE_PATH)  # add sheet_name="Sheet1"
else:
    df = pd.read_csv(FILE_PATH)

# Normalize possible column name variations to "image_url"
if "image_url" not in df.columns and "img_url" in df.columns:
    df = df.rename(columns={"img_url": "image_url"})

if "image_url" not in df.columns:
    raise ValueError("The input file must contain an 'image_url' (or 'img_url') column.")

# Retain only rows with valid HTTP(S) image URLs
df = df[df["image_url"].astype(str).str.startswith("http")].copy()
df.reset_index(drop=True, inplace=True)

# Sample 20 rows as required by Q3
if SAMPLE_METHOD == "random":
    df20 = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
else:
    df20 = df.head(SAMPLE_SIZE).copy().reset_index(drop=True)

print(f"Total rows with image_url: {len(df)} | Selected for analysis: {len(df20)}")

# Initialize AWS Rekognition client
rekognition = boto3.client("rekognition")


# Analyze each image and collect outputs
results = []

for i, row in df20.iterrows():
    image_url = str(row["image_url"])
    print(f"\n[{i+1}/{len(df20)}] {image_url}")

    # Default output fields for this row
    status = "ok"
    error = None
    labels_json = "[]"
    top_label = None
    top_confidence = None

    try:
        # Retrieve image bytes from the source URL
        headers = {"User-Agent": "Mozilla/5.0 (SageMaker-Notebook)"}  # some hosts require a User-Agent
        resp = requests.get(image_url, headers=headers, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        image_bytes = resp.content

        # Call Rekognition DetectLabels
        rk = rekognition.detect_labels(Image={"Bytes": image_bytes})
        labels = rk.get("Labels", [])

        # Determine highest-confidence label; retain labels meeting the threshold
        filtered = []
        max_name, max_conf = None, -1.0

        for lab in labels:
            name = lab.get("Name")
            conf = float(lab.get("Confidence", 0.0))
            if conf > max_conf:
                max_name, max_conf = name, conf
            if conf >= CONF_THRESHOLD:
                filtered.append({"name": name, "confidence": conf})

        top_label = max_name
        top_confidence = max_conf if max_conf >= 0 else None
        labels_json = json.dumps(filtered, ensure_ascii=False)

    except Exception as e:
        status = "error"
        error = str(e)[:300]

    # Combine original dataset columns with newly derived fields
    row_out = row.to_dict()
    row_out.update({
        "top_label": top_label,
        "top_confidence": top_confidence,
        "labels_json": labels_json,   # JSON array of labels with confidence >= threshold
        "status": status,
        "error": error,
    })
    results.append(row_out)

    # Insert a short delay between requests to prevent excessive load on servers
    time.sleep(random.uniform(*SLEEP_BETWEEN))

# Write results to Excel
out_df = pd.DataFrame(results)
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    out_df.to_excel(writer, index=False, sheet_name="analysis")

print(f"\nExported analysis results to: {OUTPUT_XLSX}")
out_df.head()


Total rows with image_url: 75 | Selected for analysis: 20

[1/20] https://i.pinimg.com/236x/73/82/78/7382781b71009c6a37c22f15a0216fd1.jpg

[2/20] https://i.pinimg.com/236x/14/a3/d9/14a3d99bc9866bdaff29f9c746772811.jpg

[3/20] https://i.pinimg.com/236x/3a/51/48/3a5148ce1f402f7d0c988def357a79a5.jpg

[4/20] https://i.pinimg.com/236x/ab/f2/80/abf280cbef5c88002088de60ea42fdb7.jpg

[5/20] https://i.pinimg.com/236x/c9/1b/bb/c91bbbc1441200f1e9f80ba243507bd5.jpg

[6/20] https://i.pinimg.com/236x/32/7e/cb/327ecb53e08f289ff3255a061afce966.jpg

[7/20] https://i.pinimg.com/236x/06/d8/33/06d83315c8e8ce9269c65ce4783e85e1.jpg

[8/20] https://i.pinimg.com/236x/a0/f6/f8/a0f6f8b043fca073147fcb0f16367470.jpg

[9/20] https://i.pinimg.com/236x/81/55/91/8155910117367bfc5a4723fe59e7411b.jpg

[10/20] https://i.pinimg.com/236x/e1/89/ef/e189efd3419a259437e6378784a5b7b4.jpg

[11/20] https://i.pinimg.com/236x/c7/f9/9b/c7f99b17321dc7d29244dd45fe6183df.jpg

[12/20] https://i.pinimg.com/236x/61/d1/c6/61d1c65e31ea639c

,id,pin_url,image_url,image_filename,pin_title_or_alt,pinner_account,board_or_topic,saves_count,comments_count,reactions_count,repins_or_shares_count,date_captured,why_it_caught_my_eye,status,top_label,top_confidence,labels_json,error
0,1,https://in.pinterest.com/pin/4081455907315965/,https://i.pinimg.com/236x/73/82/78/7382781b710...,001_theme.jpg,Home Decor with indoor plants and fairy lights,LawsonYVH,lawson |Home Decore,NaN,0,317,NaN,2025-09-19,I liked the cozy aesthetic of the living room ...,ok,Architecture,99.999985,"[{""name"": ""Architecture"", ""confidence"": 99.999...",None
1,2,https://in.pinterest.com/pin/10766486606563151/,https://i.pinimg.com/236x/14/a3/d9/14a3d99bc98...,002_theme.jpg,5 Indian Living Room Ideas You’ll Want to Share!,MyDecordesignIdeas,Home & Kitchen Design and Decoration Ideas,NaN,0,255,NaN,2025-09-19,I liked how the room combines traditional Indi...,ok,Indoors,99.998947,"[{""name"": ""Indoors"", ""confidence"": 99.99894714...",None
2,3,https://in.pinterest.com/pin/297941331621153810/,https://i.pinimg.com/236x/3a/51/48/3a5148ce1f4...,003_theme.jpg,Entry way makeover,mallammaqa,NaN,NaN,0,159,NaN,2025-09-19,The fairy lights and warm tones make the room ...,ok,Altar,99.996696,"[{""name"": ""Altar"", ""confidence"": 99.9966964721...",None
3,4,https://in.pinterest.com/pin/1477812374087497/,https://i.pinimg.com/236x/ab/f2/80/abf280cbef5...,004_theme.jpg,Home Decor Living Room,thevisualvibes,NaN,NaN,4,1600,NaN,2025-09-19,I liked the modern decor and how plants bright...,ok,Architecture,99.999428,"[{""name"": ""Architecture"", ""confidence"": 99.999...",None
4,5,https://in.pinterest.com/pin/985231164849089/,https://i.pinimg.com/236x/c9/1b/bb/c91bbbc1441...,005_theme.jpg,Cozy living room with soft lighting,espindolajohannaelizabeth,NaN,NaN,0,308,NaN,2025-09-19,The warm earthy tones and candle centerpiece m...,ok,Home Decor,99.999153,"[{""name"": ""Home Decor"", ""confidence"": 99.99915...",None


In this task, I used Amazon SageMaker with a Notebook Instance to analyze 20 images scraped in Question 1. I began by loading the dataset (214_Q1.xlsx) that contained 75 Pinterest image links and then selected 20 of them for analysis. 

For each image, I fetched the file directly from its URL and passed the image bytes into AWS Rekognition’s DetectLabels API. Rekognition returned object and scene labels along with confidence scores. From these results, I captured the top label (the one with the highest confidence) and also saved all labels above a 90% confidence threshold into a JSON list. 

To ensure the process was reliable, I included status and error fields so any failed images would be logged. Finally, I combined these analysis results with the original metadata (pin URL, image title, pinner account, etc.) and exported everything into a structured Excel file called q3_image_analysis.xlsx. 

The output shows, for each of the 20 images, the original pin details alongside Rekognition’s top predicted label, its confidence score, and the list of other high-confidence labels.

In [9]:
import json
import pandas as pd

INFILE   = "q3_image_analysis.xlsx"       # current export from analysis step
SHEET    = 0                              
OUT_XLSX = "q3_image_analysis_final.xlsx" # final polished file

# Load
df = pd.read_excel(INFILE, sheet_name=SHEET)

# Date: keep only ONE column named 'date_captured' in ISO (YYYY-MM-DD)
if "date_captured_iso" in df.columns:
    df["date_captured"] = pd.to_datetime(df["date_captured_iso"], errors="coerce").dt.strftime("%Y-%m-%d")
elif "date_captured" in df.columns:
    df["date_captured"] = pd.to_datetime(df["date_captured"], errors="coerce").dt.strftime("%Y-%m-%d")

# Drop extra date_captured_iso column if it exists
if "date_captured_iso" in df.columns:
    df = df.drop(columns=["date_captured_iso"])

# Confidence: rounded to whole percent (0–100) 
if "top_confidence" in df.columns:
    df["top_confidence"] = pd.to_numeric(df["top_confidence"], errors="coerce")
    df["top_confidence_pct"] = (
        df["top_confidence"].round(0).clip(lower=0, upper=100).astype("Int64")
    )
else:
    df["top_confidence_pct"] = pd.NA

# --- Top 3 labels (string) from labels_json ---
def mk_top3(row):
    try:
        items = json.loads(row.get("labels_json") or "[]")
        parts = [
            f'{d.get("name")} ({round(float(d.get("confidence", 0)),1)}%)'
            for d in items[:3]
        ]
        return "; ".join(parts) if parts else ""
    except Exception:
        return ""

if "labels_json" in df.columns and "top3_labels" not in df.columns:
    df["top3_labels"] = df.apply(mk_top3, axis=1)

# Column order: keep all, show key fields first
preferred = [
    "id","pin_url","image_url","image_filename","pin_title_or_alt","pinner_account",
    "board_or_topic","saves_count","comments_count","reactions_count","repins_or_shares_count",
    "date_captured","top_label","top_confidence_pct","top3_labels","labels_json","status","error"
]
ordered = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
df = df[ordered]

# Save final polished file
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
    df.to_excel(w, index=False, sheet_name="clean")

print(f"✅ Wrote {OUT_XLSX} with a single 'date_captured' column.")
df.head(3)


✅ Wrote q3_image_analysis_final.xlsx with a single 'date_captured' column.


,id,pin_url,image_url,image_filename,pin_title_or_alt,pinner_account,board_or_topic,saves_count,comments_count,reactions_count,repins_or_shares_count,date_captured,top_label,top_confidence_pct,top3_labels,labels_json,status,error,why_it_caught_my_eye,top_confidence
0,1,https://in.pinterest.com/pin/4081455907315965/,https://i.pinimg.com/236x/73/82/78/7382781b710...,001_theme.jpg,Home Decor with indoor plants and fairy lights,LawsonYVH,lawson |Home Decore,NaN,0,317,NaN,2025-09-19,Architecture,100,Architecture (100.0%); Building (100.0%); Furn...,"[{""name"": ""Architecture"", ""confidence"": 99.999...",ok,NaN,I liked the cozy aesthetic of the living room ...,99.999985
1,2,https://in.pinterest.com/pin/10766486606563151/,https://i.pinimg.com/236x/14/a3/d9/14a3d99bc98...,002_theme.jpg,5 Indian Living Room Ideas You’ll Want to Share!,MyDecordesignIdeas,Home & Kitchen Design and Decoration Ideas,NaN,0,255,NaN,2025-09-19,Indoors,100,Indoors (100.0%); Reception (100.0%); Receptio...,"[{""name"": ""Indoors"", ""confidence"": 99.99894714...",ok,NaN,I liked how the room combines traditional Indi...,99.998947
2,3,https://in.pinterest.com/pin/297941331621153810/,https://i.pinimg.com/236x/3a/51/48/3a5148ce1f4...,003_theme.jpg,Entry way makeover,mallammaqa,NaN,NaN,0,159,NaN,2025-09-19,Altar,100,Altar (100.0%); Architecture (100.0%); Buildin...,"[{""name"": ""Altar"", ""confidence"": 99.9966964721...",ok,NaN,The fairy lights and warm tones make the room ...,99.996696


After generating the initial analysis results, I created a second script to clean and polish the output. First, I standardized the date information so that only one column, date_captured, was kept and formatted in ISO style (YYYY-MM-DD). 

I then refined the confidence values by rounding them to whole percentages for easier readability and added a new field top_confidence_pct. To make the Rekognition labels more interpretable, I created a top3_labels column that summarizes the three highest-confidence labels for each image in a simple text format. Finally, I reorganized the columns so that key fields (such as image details, date, top label, and top confidence) appear first, while still preserving all other original data. 

The result was saved as q3_image_analysis_final.xlsx, a single polished file that is clean, standardized, and ready for submission.

In [10]:
import pandas as pd

IN_XLSX = "q3_image_analysis_final.xlsx"   # the polished file I created
OUT_CSV = "q3_image_analysis_final.csv"

df = pd.read_excel(IN_XLSX, sheet_name=0)

df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"CSV written: {OUT_CSV} with {len(df)} rows")


CSV written: q3_image_analysis_final.csv with 20 rows


In the final step, I converted my polished Excel file into the required CSV format for submission. I loaded the cleaned results from q3_image_analysis_final.xlsx and exported them directly to q3_image_analysis_final.csv using UTF-8 encoding to ensure special characters (such as symbols or emojis in pin titles) were preserved correctly. 

This produced a single CSV file with 20 rows of analyzed images, including all original metadata along with the standardized date, top confidence percentage, and top-3 labels.